# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: Feature Importance (Random Forest → Health Score)**

**The Paper's Claim:** In the ML Appendix, a Random Forest predicts `Health Score` with 43% importance placed on Average Position, 32% on Impressions, and 15% on Scroll Depth.  

**Methodology Question (Label-Derived Features):**
The paper defines the `Health Score` label as a mathematical composite of Impressions (30 pts), Position (30 pts), CTR (20 pts), and Scroll Depth (20 pts). By feeding those exact metrics into the model as features to predict the score, the model suffers from strict label leakage—it is simply learning the scoring formula rather than discovering underlying patterns. While the paper correctly notes that this is "descriptive rather than causal", a rigorous validation design would strictly remove these sibling columns to see what outside features (like word count or age) actually predict a healthy page.

**Finding 2: What Predicts Growth? (Logistic Regression)**

**The Paper's Claim:**
A Logistic Regression model achieves 71% holdout accuracy when classifying growing versus declining pages, identifying content age as a negative signal and days visible as a positive signal.  

**Methodology Question (Honest Splits):**
The methodology section states this model used an "80/20 split". Does this validation design support the claim? Because "growth" is a time-series concept (30-day trend) and the dataset contains multiple pages from the same clients, a random 80/20 split risks leaking a client's specific site-wide momentum into the training set. To fully trust the 71% accuracy, we must ask if the split was time-aware (training on the past, testing on the future) or grouped by client to ensure the model isn't just memorizing specific high-performing domains.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [4]:
# Load data
import pandas as pd
from datasets import load_dataset
from google.colab import userdata

hf_token = userdata.get('HF_TOKEN')

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    split="train",
    streaming=True,
    token=hf_token
)

records = []
for i, row in enumerate(ds):
    if str(row.get('report_date', '')).startswith('2025-02'):
        records.append(row)
    if len(records) >= 10000:
        break

df_clean = pd.DataFrame(records)
df_clean['report_date'] = pd.to_datetime(df_clean['report_date'])

df_clean = df_clean.dropna(subset=['gsc_avg_position', 'gsc_impressions', 'gsc_clicks']).copy()
df_clean = df_clean[(df_clean['gsc_avg_position'] > 0) & (df_clean['gsc_impressions'] > 0)]
df_clean['ctr'] = (df_clean['gsc_clicks'] / df_clean['gsc_impressions']) * 100

print(f"Data successfully loaded with {len(df_clean)} rows ready for modeling.")

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Data successfully loaded with 9968 rows ready for modeling.


In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

df_clean['label_needs_fix'] = ((df_clean['gsc_avg_position'] <= 10) & (df_clean['ctr'] <= 2.0)).astype(int)

features = ['gsc_impressions', 'ga4_sessions', 'sessions_organic', 'sessions_direct']
X = df_clean[features].fillna(0)
y = df_clean['label_needs_fix']
groups = df_clean['client_hash_id']

def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Random naive split (Dishonest)
X_train_rand, X_test_rand, y_train_rand, y_test_rand = train_test_split(
    X, y, test_size=0.2, random_state=42
)

rf_naive = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
rf_naive.fit(X_train_rand, y_train_rand)
naive_scores = rf_naive.predict_proba(X_test_rand)[:, 1]
naive_p_k = precision_at_k(naive_scores, y_test_rand, k=50)

# Grouped split by client (Honest)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))

X_train_grp, y_train_grp = X.iloc[train_idx], y.iloc[train_idx]
X_test_grp, y_test_grp = X.iloc[test_idx], y.iloc[test_idx]

rf_honest = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=42)
rf_honest.fit(X_train_grp, y_train_grp)
honest_scores = rf_honest.predict_proba(X_test_grp)[:, 1]
honest_p_k = precision_at_k(honest_scores, y_test_grp, k=50)

# Display memorization gap
print("--- Split Design Comparison (Precision@50) ---")
print(f"Base Rate (Test Set):       {y_test_grp.mean():.3f}")
print(f"Naive Random Split Score:   {naive_p_k:.3f} (Inflated by group memorization)")
print(f"Honest Grouped Split Score: {honest_p_k:.3f} (True generalization to unseen clients)")

--- Split Design Comparison (Precision@50) ---
Base Rate (Test Set):       0.388
Naive Random Split Score:   0.480 (Inflated by group memorization)
Honest Grouped Split Score: 0.400 (True generalization to unseen clients)


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**My Model Leakage Audit:**

* **Label-derived features:**
Passed. The target label (`label_needs_fix`) was mathematically derived from `gsc_avg_position` and `ctr`. I strictly excluded both of those columns from the `X` feature set. If I had included `gsc_avg_position`, it would have been a direct Taxonomy #1 leakage violation.  

* **Future/overlapping windows:**
Passed, but with constraints. Because we are analyzing a daily-grain snapshot, our features (daily impressions) are concurrent with our labels (daily rank/CTR). In a true production environment predicting future fixes, we would need to shift the target label forward by one day to ensure the features are strictly knowable before the prediction window.

* **Decision-derived features:**
Passed. No internal FlyRank product flags (like `is_declining_label`) were used as inputs. Sanity-check on top feature: The model leans 100% on gsc_impressions. As discovered in the Week 5 error analysis, this is not a mathematical leak, but rather an artifact of the early 2025 dataset where all GA4 session features are naturally zero-filled.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Bold Claim:**

"My Random Forest model accurately identifies exactly which pages need a CTR fix because it correctly learned that high impressions guarantee a page-one ranking, proving we should always target these specific URLs."

**Rewritten:**

"The grouped validation design provides directional decision-support for identifying pages that may require a CTR fix. In the observed sample, impression volume was measured as the primary splitting feature, though this reliance is heavily influenced by the lack of variance in early GA4 data. The model helps prioritize reviews, but manual triage is still required to filter out unfixable zero-click searches."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.